In [ ]:
# Cell 1 - run once, loading data is slow
import pandas as pd
raw_business = pd.read_parquet("../../data/business_y_removed.parquet")
raw_checkin = pd.read_parquet("../../data/checkin.parquet")
raw_review = pd.read_parquet("../../data/review-001.parquet")
raw_tips = pd.read_parquet("../../data/tip.parquet")
raw_user = pd.read_parquet("../../data/user-002.parquet")


In [ ]:
import numpy as np

In [ ]:
pd.set_option('display.max_columns', None)

CLEANING RAW DATAFRAMES TO REMOVE NON-FOOD RELATED PLACES

In [ ]:
food_keywords = [
    'Restaurant', 'Food', 'Cafe', 'Coffee', 'Bakery', 
    'Bar', 'Pub', 'Diner', 'Eatery', 'Bistro', 
    'Brewery', 'Pizzeria', 'Sushi', 'Steakhouse', 'Deli'
]
pattern = '|'.join(food_keywords)
business = raw_business[raw_business['categories'].str.contains(pattern, na=False, case=False)]
business = business.dropna(subset=['is_open'])
business = business.reset_index(drop=True)
business['is_open'].value_counts(normalize=True)

In [ ]:
restaurant_ids = set(business["business_id"])
checkins = raw_checkin[raw_checkin['business_id'].isin(restaurant_ids)]
reviews = raw_review[raw_review['business_id'].isin(restaurant_ids)]
tips = raw_tips[raw_tips['business_id'].isin(restaurant_ids)]


restaurant_user_ids = set(reviews['user_id'])
users = raw_user[raw_user['user_id'].isin(restaurant_user_ids)]

----------------------------------------------------------------------------------------------------------------------------------------------------------------

CHECKING BEFORE AND AFTER SHAPES OF DATAFRAMES

In [ ]:
raw_business.shape, raw_checkin.shape, raw_review.shape, raw_tips.shape, raw_user.shape

In [ ]:
business.shape, checkins.shape, reviews.shape, tips.shape, users.shape

----------------------------------------------------------------------------------------------------------------------------------------------------------------

CHECKING CORRELATION OF VARIOUS SERIES WITH IS_OPEN ATTRIBUTE

In [ ]:
from datetime import datetime

# Convert date to datetime
reviews['date'] = pd.to_datetime(reviews['date'])

# Reference point for recency calculations
reference_date = reviews['date'].max()

# Aggregate all review features per business
review_features = reviews.groupby('business_id').agg(
    review_count_actual   = ('review_id', 'count'),
    avg_review_stars      = ('stars', 'mean'),
    std_review_stars      = ('stars', 'std'),        # consistency of ratings
    total_useful          = ('useful', 'sum'),
    total_funny           = ('funny', 'sum'),
    total_cool            = ('cool', 'sum'),
    first_review_date     = ('date', 'min'),
    last_review_date      = ('date', 'max'),
).reset_index()

# Recency — days since last review
review_features['days_since_last_review'] = (
    reference_date - review_features['last_review_date']
).dt.days

# Lifespan — how long has this business been reviewed
review_features['review_lifespan_days'] = (
    review_features['last_review_date'] - review_features['first_review_date']
).dt.days

# Velocity — average reviews per month
review_features['reviews_per_month'] = (
    review_features['review_count_actual'] / 
    (review_features['review_lifespan_days'] / 30)
).replace([float('inf')], 0)  # handle businesses with only 1 review

# Drop raw date columns — model doesn't need them
review_features = review_features.drop(
    columns=['first_review_date', 'last_review_date']
)

temp = review_features[['business_id', 'days_since_last_review']].merge(
    business[['business_id', 'is_open']], 
    on='business_id'
)

# Correlation
print(temp[['days_since_last_review', 'is_open']].corr()['is_open'])

In [ ]:
# Parse the comma-separated timestamps
checkins['date'] = checkins['date'].str.split(', ')
checkins['date'] = checkins['date'].apply(
    lambda x: pd.to_datetime(x, errors='coerce')
)

# Now engineer features
reference_date = pd.Timestamp(reviews['date'].max())

checkin_features = checkins.apply(lambda row: pd.Series({
    'business_id'              : row['business_id'],
    'total_checkins'           : len(row['date']),
    'days_since_last_checkin'  : (reference_date - max(row['date'])).days,
    'checkin_lifespan_days'    : (max(row['date']) - min(row['date'])).days,
    'checkins_per_month'       : len(row['date']) / 
                                 max((max(row['date']) - min(row['date'])).days / 30, 1)
}), axis=1)

temp2 = checkin_features[['business_id', 'days_since_last_checkin']].merge(
    business[['business_id', 'is_open']],
    on='business_id'
)

print(temp2[['days_since_last_checkin', 'is_open']].corr()['is_open'])

In [ ]:
tips['date'] = pd.to_datetime(tips['date'])

tip_features = tips.groupby('business_id').agg(
    total_tips            = ('user_id', 'count'),
    total_compliments     = ('compliment_count', 'sum'),
    days_since_last_tip   = ('date', lambda x: (reference_date - x.max()).days),
    tip_lifespan_days     = ('date', lambda x: (x.max() - x.min()).days),
).reset_index()

tip_features['tips_per_month'] = (
    tip_features['total_tips'] /
    (tip_features['tip_lifespan_days'] / 30)
).replace([float('inf')], 0)

temp3 = tip_features[['business_id', 'days_since_last_tip']].merge(
    business[['business_id', 'is_open']],
    on='business_id'
)

print(temp3[['days_since_last_tip', 'is_open']].corr()['is_open'])

In [ ]:
# Parse elite — count how many years they've been elite
users['elite_years'] = users['elite'].apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) and x != '' and x != 'None' else 0
)

# Parse yelping_since to get user experience in days
users['yelping_since'] = pd.to_datetime(users['yelping_since'])
users['yelping_experience_days'] = (reference_date - users['yelping_since']).dt.days

# Keep only what we need
user_features = users[[
    'user_id', 'elite_years', 'fans', 
    'useful', 'review_count', 'yelping_experience_days'
]]

# Join users onto reviews
reviews_with_users = reviews[['business_id', 'user_id']].merge(
    user_features, on='user_id', how='left'
)

# Aggregate per business
business_user_features = reviews_with_users.groupby('business_id').agg(
    avg_reviewer_elite_years    = ('elite_years', 'mean'),
    pct_elite_reviewers         = ('elite_years', lambda x: (x > 0).mean()),
    avg_reviewer_fans           = ('fans', 'mean'),
    avg_reviewer_useful         = ('useful', 'mean'),
    avg_reviewer_experience     = ('yelping_experience_days', 'mean'),
    unique_reviewers            = ('user_id', 'nunique'),
).reset_index()

temp4 = business_user_features[['business_id', 'avg_reviewer_fans', 
                                  'avg_reviewer_experience',
                                  'unique_reviewers']].merge(
    business[['business_id', 'is_open']],
    on='business_id'
)

print(temp4[['avg_reviewer_fans', 'avg_reviewer_experience', 
             'unique_reviewers', 'is_open']].corr()['is_open'])

----------------------------------------------------------------------------------------------------------------------------------------------------------------

REMOVING NON-FOOD RELATED ATTRIBUTES FROM BUSINESS AND FINDING TOP ATTRIBUTES

In [ ]:
import ast

def parse_attributes(attr):
    if attr is None or attr == 'None':
        return {}
    if isinstance(attr, str):
        try:
            return ast.literal_eval(attr)
        except:
            return {}
    return attr

business['attributes_parsed'] = business['attributes'].apply(parse_attributes)
print(f"After parse_attributes: {business.shape}")

In [ ]:
bool_attrs = [
    'BikeParking', 'BusinessAcceptsCreditCards', 'Caters',
    'DogsAllowed', 'DriveThru', 'GoodForKids', 
    'HappyHour', 'HasTV', 'OutdoorSeating', 'RestaurantsDelivery', 'RestaurantsGoodForGroups',
    'RestaurantsReservations', 'RestaurantsTableService', 'RestaurantsTakeOut', 
    'WheelchairAccessible'
]

def parse_bool(val):
    if val is None or val == 'None':
        return np.nan
    if isinstance(val, bool):
        return int(val)
    if isinstance(val, str):
        return 1 if val.strip().lower() == 'true' else 0
    return np.nan

for attr in bool_attrs:
    business[f'attr_{attr}'] = business['attributes_parsed'].apply(
        lambda x: parse_bool(x.get(attr))
    )

business['attr_PriceRange'] = business['attributes_parsed'].apply(
    lambda x: int(x['RestaurantsPriceRange2']) 
    if x.get('RestaurantsPriceRange2') not in [None, 'None'] 
    else np.nan
)
print(f"After bool and numeric attrs: {business.shape}")

In [ ]:
cat_attrs = ['Alcohol', 'WiFi', 'NoiseLevel', 'RestaurantsAttire']

def clean_cat(val):
    if val is None or val == 'None':
        return np.nan
    val = str(val).strip()
    val = val.replace("u'", "").replace("'", "")
    return val.lower()

for attr in cat_attrs:
    cleaned = business['attributes_parsed'].apply(
        lambda x: clean_cat(x.get(attr))
    )
    dummies = pd.get_dummies(cleaned, prefix=f'attr_{attr}', dummy_na=False).astype(int)
    business = pd.concat([business, dummies], axis=1)

business = business.loc[:, ~business.columns.duplicated()]
print(f"After categorical attrs: {business.shape}")

In [ ]:
open_restaurants = business[business['is_open'] == 1]
closed_restaurants = business[business['is_open'] == 0]

attr_cols = [c for c in business.columns if c.startswith('attr_')]

ratio_analysis = pd.DataFrame({
    'open_rate'   : open_restaurants[attr_cols].mean(),
    'closed_rate' : closed_restaurants[attr_cols].mean(),
})

ratio_analysis['difference'] = ratio_analysis['open_rate'] - ratio_analysis['closed_rate']
print(ratio_analysis.sort_values('difference', ascending=False).to_string())

MERGING INTO A FINAL DATAFRAME

In [ ]:
final_df = business[['business_id', 'is_open', 'stars', 'review_count', 
                      'latitude', 'longitude'] + attr_cols].copy()

final_df = final_df.merge(review_features, on='business_id', how='left')
final_df = final_df.merge(checkin_features, on='business_id', how='left')
final_df = final_df.merge(tip_features, on='business_id', how='left')
final_df = final_df.merge(business_user_features, on='business_id', how='left')

print(final_df.shape)
print(final_df.isnull().sum())

In [ ]:
final_df.head()

In [ ]:
# Separate features from target
attr_cols = [c for c in final_df.columns if c.startswith('attr_')]

X = final_df[attr_cols + ['stars', 'review_count', 'latitude', 'longitude']]
y = final_df['is_open']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric='auc',
    enable_categorical=False
)

model.fit(X_train, y_train)
print("Training complete")

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(classification_report(y_test, y_pred))

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=20)

CROSS VALIDATION TESTING

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
print(scores)
print(f"Mean AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")